In [1]:
import json
import os
from pathlib import Path
from IPython.display import display, HTML, Image
import pandas as pd

In [2]:
# Paths
COREF_DIR = Path("../models/linkappend/data-out/conll_to_json")
IMAGES_DIR = Path("../data/sampled_60/images")
CHARACTERS_DIR = Path("../data/sampled_60/characters")
METADATA_CSV = Path("../data/vwp-acl2025-subset.csv")

# Load metadata to get story IDs and image paths
metadata = pd.read_csv(METADATA_CSV)

In [3]:
def load_jsonlines(filepath):
    """Load jsonlines file and return list of documents."""
    documents = []
    with open(filepath, 'r') as f:
        for line in f:
            documents.append(json.loads(line))
    return documents

def get_story_text(doc):
    """Convert sentences to full text."""
    return ' '.join([' '.join(sent) for sent in doc['sentences']])

def highlight_coref_chains(doc):
    """Create HTML with coreference chains highlighted, separating each sentence/description."""
    # Assign colors to clusters
    colors = ['#FFB3BA', '#BAFFC9', '#BAE1FF', '#FFFFBA', '#FFDFBA', '#E0BBE4', '#FFDFD3']
    cluster_colors = {}
    for i, cluster in enumerate(doc['clusters']):
        cluster_colors[i] = colors[i % len(colors)]
    
    # Process each sentence separately
    html_sentences = []
    token_offset = 0  # Track global token position
    
    for sent_idx, sent in enumerate(doc['sentences']):
        # Mark tokens with cluster membership for this sentence
        token_clusters = {}
        for cluster_id, cluster in enumerate(doc['clusters']):
            for mention in cluster:
                start, end = mention[0], mention[1]
                # Check if mention overlaps with current sentence
                if start < token_offset + len(sent) and end >= token_offset:
                    for pos in range(max(start, token_offset), min(end + 1, token_offset + len(sent))):
                        local_pos = pos - token_offset
                        if local_pos not in token_clusters:
                            token_clusters[local_pos] = []
                        token_clusters[local_pos].append(cluster_id)
        
        # Build HTML for this sentence
        html_tokens = []
        for pos, token in enumerate(sent):
            if pos in token_clusters:
                cluster_id = token_clusters[pos][0]
                color = cluster_colors[cluster_id]
                html_tokens.append(f'<span style="background-color: {color}; padding: 2px; border-radius: 3px;">{token}</span>')
            else:
                html_tokens.append(token)
        
        # Add sentence as a paragraph
        html_sentences.append(f'<p>{" ".join(html_tokens)}</p>')
        token_offset += len(sent)
    
    return '\n'.join(html_sentences)

def display_story_with_images(doc, story_id):
    """Display story with images and coreference highlighting."""
    # Display header
    display(HTML(f"<h2>Story {story_id}</h2>"))
    
    # Get image URLs from metadata
    story_meta = metadata[metadata['story_id'] == int(story_id)]
    
    if not story_meta.empty:
        # Display images from local files for notebook
        display(HTML("<h3>Images</h3>"))
        image_html = '<div style="display: flex; flex-wrap: wrap; gap: 10px;">'
        
        # Get local image files
        image_files = sorted([f for f in os.listdir(IMAGES_DIR) if f.startswith(f"{story_id}_img") and f.endswith('.jpg')])
        for img_file in image_files:
            img_path = IMAGES_DIR / img_file
            image_html += f'<img src="{img_path}" width="150" style="border: 1px solid #ddd; border-radius: 4px;">'
        
        image_html += '</div>'
        display(HTML(image_html))
        
        # Display character images from local files
        char_files = sorted([f for f in os.listdir(CHARACTERS_DIR) if f.startswith(f"{story_id}_char") and f.endswith('.jpg')])
        
        if char_files:
            display(HTML("<h3>Characters</h3>"))
            char_html = '<div style="display: flex; flex-wrap: wrap; gap: 10px;">'
            for char_file in char_files:
                char_path = CHARACTERS_DIR / char_file
                char_html += f'<img src="{char_path}" width="100" style="border: 1px solid #ddd; border-radius: 4px;">'
            char_html += '</div>'
            display(HTML(char_html))
    
    # Display story with coreference highlighting
    display(HTML("<h3>Story with Coreference Chains</h3>"))
    highlighted_text = highlight_coref_chains(doc)
    display(HTML(f'<div style="padding: 15px; background-color: #f9f9f9; border-radius: 5px; line-height: 1.8;">{highlighted_text}</div>'))
    
    # Display coreference chain summary
    display(HTML("<h3>Coreference Chains</h3>"))
    for i, cluster in enumerate(doc['clusters']):
        mentions = []
        tokens = [token for sent in doc['sentences'] for token in sent]
        for mention in cluster:
            start, end = mention[0], mention[1]
            mention_text = ' '.join(tokens[start:end+1])
            mentions.append(mention_text)
        display(HTML(f"<b>Chain {i+1}:</b> {' → '.join(mentions)}"))
    
    display(HTML("<hr>"))

In [4]:
# List available files
jsonlines_files = sorted([f for f in os.listdir(COREF_DIR) if f.endswith('.jsonlines')])
print("Available files:")
for i, f in enumerate(jsonlines_files):
    print(f"{i}: {f}")

Available files:
0: claude45_large_seed42_test_snapshots__local_json-nopound_pred.jsonlines
1: claude45_large_seed43_test_snapshots__local_json-nopound_pred.jsonlines
2: claude45_large_seed44_test_snapshots__local_json-nopound_pred.jsonlines
3: claude45_original_seed44_test_snapshots__local_json-nopound_pred.jsonlines
4: gpt4o_large_seed42_test_snapshots__local_json-nopound_pred.jsonlines
5: gpt4o_large_seed43_test_snapshots__local_json-nopound_pred.jsonlines
6: gpt4o_large_seed44_test_snapshots__local_json-nopound_pred.jsonlines
7: gpt4o_original_seed42_test_snapshots__local_json-nopound_pred.jsonlines
8: human_large_seed42_test_snapshots__local_json-nopound_pred.jsonlines
9: human_large_seed43_test_snapshots__local_json-nopound_pred.jsonlines
10: human_large_seed44_test_snapshots__local_json-nopound_pred.jsonlines
11: human_original_seed42_test_snapshots__local_json-nopound_pred.jsonlines
12: internvl3_large_seed42_test_snapshots__local_json-nopound_pred.jsonlines
13: internvl3_large

In [5]:
# Select a file (change the index to view different models)
file_index = 11  # Change this to select different file
selected_file = jsonlines_files[file_index]
print(f"Viewing: {selected_file}")

# Load documents
documents = load_jsonlines(COREF_DIR / selected_file)
print(f"Loaded {len(documents)} stories")

Viewing: human_original_seed42_test_snapshots__local_json-nopound_pred.jsonlines
Loaded 60 stories


In [6]:
# Display first 3 stories (change range to view more)
for doc in documents[:3]:
    story_id = doc['doc_key']
    display_story_with_images(doc, story_id)

In [7]:
# Enter a specific story ID to view
target_story_id = "2926"  # Change this to view different story

# Find and display the story
for doc in documents:
    if doc['doc_key'] == target_story_id:
        display_story_with_images(doc, target_story_id)
        break
else:
    print(f"Story {target_story_id} not found")

In [8]:
# Calculate statistics
total_stories = len(documents)
total_chains = sum(len(doc['clusters']) for doc in documents)
avg_chains_per_story = total_chains / total_stories if total_stories > 0 else 0

chain_lengths = []
for doc in documents:
    for cluster in doc['clusters']:
        chain_lengths.append(len(cluster))

avg_chain_length = sum(chain_lengths) / len(chain_lengths) if chain_lengths else 0

print(f"Total stories: {total_stories}")
print(f"Total coreference chains: {total_chains}")
print(f"Average chains per story: {avg_chains_per_story:.2f}")
print(f"Average chain length (mentions): {avg_chain_length:.2f}")

Total stories: 60
Total coreference chains: 228
Average chains per story: 3.80
Average chain length (mentions): 4.30


In [10]:
def generate_story_html(doc, story_id):
    """Generate HTML for a single story visualization."""
    html_parts = []
    
    # Header
    html_parts.append(f"<html><head><meta charset='utf-8'><title>Story {story_id}</title>")
    html_parts.append("<style>")
    html_parts.append("body { font-family: Arial, sans-serif; max-width: 1200px; margin: 20px auto; padding: 20px; }")
    html_parts.append("h2, h3 { color: #333; }")
    html_parts.append(".images, .characters { display: flex; flex-wrap: wrap; gap: 10px; margin: 20px 0; }")
    html_parts.append("img { border: 1px solid #ddd; border-radius: 4px; }")
    html_parts.append(".story-text { padding: 15px; background-color: #f9f9f9; border-radius: 5px; line-height: 1.8; margin: 20px 0; }")
    html_parts.append(".chain { margin: 10px 0; }")
    html_parts.append("</style></head><body>")
    
    # Story header
    html_parts.append(f"<h2>Story {story_id}</h2>")
    
    # Get metadata for this story
    story_meta = metadata[metadata['story_id'] == int(story_id)]
    
    if not story_meta.empty:
        row = story_meta.iloc[0]
        
        # Images from URLs
        html_parts.append("<h3>Images</h3>")
        html_parts.append('<div class="images">')
        for i in range(10):  # link0 through link9
            col_name = f'link{i}'
            if col_name in row and pd.notna(row[col_name]) and row[col_name]:
                html_parts.append(f'<img src="{row[col_name]}" width="150">')
        html_parts.append('</div>')
        
        # Characters from URLs
        html_parts.append("<h3>Characters</h3>")
        html_parts.append('<div class="characters">')
        for i in range(5):  # char0_url through char4_url
            col_name = f'char{i}_url'
            if col_name in row and pd.notna(row[col_name]) and row[col_name]:
                html_parts.append(f'<img src="{row[col_name]}" width="100">')
        html_parts.append('</div>')
    
    # Story with highlighting
    html_parts.append("<h3>Story with Coreference Chains</h3>")
    highlighted_text = highlight_coref_chains(doc)
    html_parts.append(f'<div class="story-text">{highlighted_text}</div>')
    
    # Coreference chains
    html_parts.append("<h3>Coreference Chains</h3>")
    for i, cluster in enumerate(doc['clusters']):
        mentions = []
        tokens = [token for sent in doc['sentences'] for token in sent]
        for mention in cluster:
            start, end = mention[0], mention[1]
            mention_text = ' '.join(tokens[start:end+1])
            mentions.append(mention_text)
        html_parts.append(f'<div class="chain"><b>Chain {i+1}:</b> {" → ".join(mentions)}</div>')
    
    html_parts.append("</body></html>")
    
    return '\n'.join(html_parts)

def save_all_visualizations(selected_file, documents, output_base_dir="../examine_stories"):
    """Save all story visualizations as HTML files."""
    # Extract model and prompt info from filename
    # e.g., "claude45_large_seed42_test_snapshots__local_json-nopound_pred.jsonlines"
    parts = selected_file.replace("_test_snapshots__local_json-nopound_pred.jsonlines", "").split("_")
    
    # Determine model and prompt folder name
    if "seed" in selected_file:
        # Has seed, e.g., "claude45_large_seed42"
        model_prompt = "_".join(parts)
    else:
        # No seed, e.g., "claude45_original"
        model_prompt = "_".join(parts)
    
    # Create output directory
    output_dir = Path(output_base_dir) / model_prompt
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Saving {len(documents)} stories to {output_dir}")
    
    # Generate and save HTML for each story
    for doc in documents:
        story_id = doc['doc_key']
        html_content = generate_story_html(doc, story_id)
        
        output_file = output_dir / f"story_{story_id}.html"
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(html_content)
    
    print(f"✓ Saved {len(documents)} HTML files to {output_dir}")
    return output_dir

In [11]:
# Save all stories from the currently selected file
output_dir = save_all_visualizations(selected_file, documents)
print(f"\nView HTML files at: {output_dir.absolute()}")

Saving 60 stories to ../examine_stories/human_original_seed42
✓ Saved 60 HTML files to ../examine_stories/human_original_seed42

View HTML files at: /mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/notebooks/../examine_stories/human_original_seed42
✓ Saved 60 HTML files to ../examine_stories/human_original_seed42

View HTML files at: /mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/notebooks/../examine_stories/human_original_seed42


In [12]:
# Process all jsonlines files and save visualizations
for jsonlines_file in jsonlines_files:
    print(f"\n{'='*60}")
    print(f"Processing: {jsonlines_file}")
    
    docs = load_jsonlines(COREF_DIR / jsonlines_file)
    save_all_visualizations(jsonlines_file, docs)

print(f"\n{'='*60}")
print("✓ All visualizations saved!")


Processing: claude45_large_seed42_test_snapshots__local_json-nopound_pred.jsonlines
Saving 60 stories to ../examine_stories/claude45_large_seed42
✓ Saved 60 HTML files to ../examine_stories/claude45_large_seed42

Processing: claude45_large_seed43_test_snapshots__local_json-nopound_pred.jsonlines
Saving 60 stories to ../examine_stories/claude45_large_seed43
✓ Saved 60 HTML files to ../examine_stories/claude45_large_seed42

Processing: claude45_large_seed43_test_snapshots__local_json-nopound_pred.jsonlines
Saving 60 stories to ../examine_stories/claude45_large_seed43
✓ Saved 60 HTML files to ../examine_stories/claude45_large_seed43

Processing: claude45_large_seed44_test_snapshots__local_json-nopound_pred.jsonlines
Saving 60 stories to ../examine_stories/claude45_large_seed44
✓ Saved 60 HTML files to ../examine_stories/claude45_large_seed43

Processing: claude45_large_seed44_test_snapshots__local_json-nopound_pred.jsonlines
Saving 60 stories to ../examine_stories/claude45_large_seed44
✓